# GridSentinel AI - run everything
Run the cells top to bottom (Runtime > Run all). Simulation on IEEE 14-bus; true voltage is used only for scoring.

In [ ]:
!pip -q install pandapower==3.5.6 scikit-learn scipy matplotlib streamlit pytest

In [ ]:
import os
os.makedirs('/content/gridsentinel', exist_ok=True)
os.chdir('/content/gridsentinel')

In [ ]:
%%writefile gridsentinel_core.py
"""
GridSentinel AI - core pipeline
================================
Trust-aware decision layer for smart-grid state estimation under false-data
injection attacks (FDIA), IEEE 14-bus system (pandapower).

DESIGN RULE (the "no oracle" rule)
----------------------------------
* `run_pipeline(model, z, cfg, calibration)` is the LIVE path. It receives only
  what a control-centre operator sees: the measurement vector `z`, the public
  network model, and an offline-calibrated gate. It never receives the true
  state, the attack label or the attacked meter list.
* Everything that needs the truth (scenario generation, attacks, error
  metrics, recovery tables) lives in clearly separated functions:
  `make_scenarios`, `apply_attack`, `recovery_table`, `evaluate_run`.

Measurement set (full): bus voltage magnitudes (14), bus P/Q injections (28),
branch P/Q flows at BOTH ends (80) = 122 measurements for 27 states.

Pipeline: robust (Huber) AC-WLS fit -> trust score per measurement from
normalized residuals with a median/MAD robust scale -> isolation with an
observability guard -> re-estimation on the trusted set -> runtime-only gate
features -> calibrated risk bound (split-conformal, Mondrian bins) ->
PROCEED / CAUTION / VALIDATE.
"""
from __future__ import annotations

import copy
import json
import math
import time
from dataclasses import dataclass, field

import numpy as np

KINDS = ("V", "Pinj", "Qinj", "Pf", "Qf", "Pt", "Qt")
FEATURE_NAMES = (
    "log_J",          # log10 of chi-square statistic per degree of freedom (retained set)
    "log_zmax",       # log10 of largest remaining robust z-score
    "frac_iso",       # fraction of measurements isolated
    "log_shift",      # log10 of max change in voltage estimate caused by recovery
    "log_disagree",   # log10 of max disagreement between WLS recovery and Huber estimate
    "log_sdV",        # log10 of model-based std of the voltage estimate (from gain matrix)
    "n_crit",         # number of critical (undetectable-error) measurements retained
    "obs_blocked",    # 1 if suspicious measurements could not be removed (observability)
    "log_obs",        # log10 of observability margin (relative smallest eigenvalue)
    "log_J_raw",      # log10 chi-square per dof of the RAW (uncleaned) WLS fit: conflict before cleaning
    "log_zfirst",     # log10 of the largest z-score seen BEFORE any isolation
)
K_MAX = 8  # threat model: attacker can compromise at most K_MAX meters
IN_MODEL_MIX = {
    "none": 0.15,
    "random_single": 0.15,
    "single_small": 0.15,
    "multi_coordinated": 0.15,
    "stealth_partial": 0.20,
    "adaptive": 0.20,
}
ATTACK_MODES = (
    "none", "random_single", "single_small", "multi_coordinated",
    "stealth_partial", "adaptive", "stealth_full", "custom_voltage", "custom_stealth",
)


# =============================================================================
# 1. Network / AC measurement model
# =============================================================================
@dataclass
class Fit:
    x: np.ndarray
    converged: bool
    iters: int


@dataclass
class Analysis:
    rN: np.ndarray        # normalized residuals (0 for critical / unavailable)
    crit: np.ndarray      # critical measurements (error undetectable)
    res: np.ndarray       # raw residuals z - h(x)
    Ginv: np.ndarray
    lam_min: float


class GridModel:
    """IEEE-14 AC measurement model h(x), Jacobian, WLS / Huber estimators."""

    def __init__(self, meas_set: str = "full", sigma_v: float = 0.004, sigma_pq: float = 1.0):
        import pandapower as pp
        import pandapower.networks as pn

        net = pn.case14()
        pp.runpp(net, calculate_voltage_angles=True, numba=False)
        ppc = net._ppc
        lk = np.asarray(net._pd2ppc_lookups["bus"])
        nb = len(net.bus)
        if not np.array_equal(lk[:nb], np.arange(nb)):
            raise RuntimeError("Unexpected bus ordering in pandapower ppc")
        internal = ppc["internal"]
        self.net = net
        self.base = float(ppc["baseMVA"])
        self.Ybus = np.asarray(internal["Ybus"].todense(), dtype=complex)
        self.Yf = np.asarray(internal["Yf"].todense(), dtype=complex)
        self.Yt = np.asarray(internal["Yt"].todense(), dtype=complex)
        br = ppc["branch"]
        self.f = br[:, 0].real.astype(int)
        self.t = br[:, 1].real.astype(int)
        self.nb = nb
        self.nl = len(self.f)
        self.slack = int(net.ext_grid.bus.iloc[0])
        self.nonref = np.array([b for b in range(nb) if b != self.slack])
        self.n_state = 2 * nb - 1
        self.Cf = np.zeros((self.nl, nb))
        self.Ct = np.zeros((self.nl, nb))
        self.Cf[np.arange(self.nl), self.f] = 1.0
        self.Ct[np.arange(self.nl), self.t] = 1.0

        self.kinds = np.array(["V"] * nb + ["Pinj"] * nb + ["Qinj"] * nb
                              + ["Pf"] * self.nl + ["Qf"] * self.nl + ["Pt"] * self.nl + ["Qt"] * self.nl)
        self.elem = np.concatenate([np.arange(nb)] * 3 + [np.arange(self.nl)] * 4)
        self.loc = np.concatenate([np.arange(nb)] * 3 + [self.f, self.f, self.t, self.t])
        self.m = len(self.kinds)
        self.sigma = np.where(self.kinds == "V", sigma_v, sigma_pq).astype(float)
        self.meas_set = meas_set
        if meas_set == "full":
            self.avail = np.ones(self.m, bool)
        elif meas_set == "basic":  # voltages + injections only (ablation)
            self.avail = np.isin(self.kinds, ["V", "Pinj", "Qinj"])
        else:
            raise ValueError("meas_set must be 'full' or 'basic'")
        self.names = []
        for k, e in zip(self.kinds, self.elem):
            if k == "V":
                self.names.append(f"V@B{e + 1}")
            elif k in ("Pinj", "Qinj"):
                self.names.append(f"{k[0]}_inj@B{e + 1}")
            else:
                a, b = (self.f[e], self.t[e]) if k.endswith("f") else (self.t[e], self.f[e])
                self.names.append(f"{k[0]}_flow B{a + 1}->B{b + 1}")
        self.iV = np.arange(nb)  # measurement indices of voltage meters
        self.stateV = (nb - 1) + np.arange(nb)  # state indices of voltage magnitudes

        V0 = net.res_bus.vm_pu.values.copy()
        th0 = np.deg2rad(net.res_bus.va_degree.values)
        self.x_base = self.pack(V0, th0)
        self.lam_ref = self.obs_lambda(self.x_base, self.avail)

    # -- state packing ---------------------------------------------------
    def pack(self, V, th):
        return np.concatenate([np.asarray(th)[self.nonref], np.asarray(V)])

    def unpack(self, x):
        th = np.zeros(self.nb)
        th[self.nonref] = x[: self.nb - 1]
        return np.asarray(x[self.nb - 1:]), th

    def flat_start(self):
        return self.pack(np.ones(self.nb), np.zeros(self.nb))

    # -- measurement function and Jacobian ---------------------------------
    def h(self, x):
        V, th = self.unpack(x)
        Vc = V * np.exp(1j * th)
        Sb = Vc * np.conj(self.Ybus @ Vc) * self.base
        Sf = Vc[self.f] * np.conj(self.Yf @ Vc) * self.base
        St = Vc[self.t] * np.conj(self.Yt @ Vc) * self.base
        return np.concatenate([V, Sb.real, Sb.imag, Sf.real, Sf.imag, St.real, St.imag])

    def jac(self, x):
        V, th = self.unpack(x)
        nb = self.nb
        Vc = V * np.exp(1j * th)
        Vn = np.exp(1j * th)
        Ib = self.Ybus @ Vc
        dV = np.diag(Vc)
        dVn = np.diag(Vn)
        dSb_dVm = dV @ np.conj(self.Ybus @ dVn) + np.conj(np.diag(Ib)) @ dVn
        dSb_dVa = 1j * dV @ np.conj(np.diag(Ib) - self.Ybus @ dV)
        If = self.Yf @ Vc
        It = self.Yt @ Vc
        dVf = np.diag(Vc[self.f])
        dVt = np.diag(Vc[self.t])
        dSf_dVa = 1j * (np.conj(np.diag(If)) @ self.Cf @ dV - dVf @ np.conj(self.Yf @ dV))
        dSf_dVm = dVf @ np.conj(self.Yf @ dVn) + np.conj(np.diag(If)) @ self.Cf @ dVn
        dSt_dVa = 1j * (np.conj(np.diag(It)) @ self.Ct @ dV - dVt @ np.conj(self.Yt @ dV))
        dSt_dVm = dVt @ np.conj(self.Yt @ dVn) + np.conj(np.diag(It)) @ self.Ct @ dVn
        nr = self.nonref
        b = self.base

        def blk(dVa, dVm, part):
            return np.hstack([part(dVa)[:, nr], part(dVm)]) * b

        HV = np.hstack([np.zeros((nb, nb - 1)), np.eye(nb)])
        return np.vstack([
            HV,
            blk(dSb_dVa, dSb_dVm, np.real), blk(dSb_dVa, dSb_dVm, np.imag),
            blk(dSf_dVa, dSf_dVm, np.real), blk(dSf_dVa, dSf_dVm, np.imag),
            blk(dSt_dVa, dSt_dVm, np.real), blk(dSt_dVa, dSt_dVm, np.imag),
        ])

    # -- estimators ----------------------------------------------------------
    def wls(self, z, active, sig_eff=None, x0=None, max_iter=30, tol=1e-6) -> Fit:
        sig = self.sigma if sig_eff is None else sig_eff
        idx = np.where(active)[0]
        w = 1.0 / sig[idx] ** 2
        zi = z[idx]
        x = self.flat_start() if x0 is None else np.array(x0, float)

        def cost(xx):
            r = zi - self.h(xx)[idx]
            return float(np.sum(w * r * r))

        c = cost(x)
        for it in range(max_iter):
            r = zi - self.h(x)[idx]
            H = self.jac(x)[idx]
            G = H.T @ (H * w[:, None])
            g = H.T @ (w * r)
            try:
                dx = np.linalg.solve(G, g)
            except np.linalg.LinAlgError:
                return Fit(x, False, it)
            step = 1.0
            xn, cn = x, c
            for _ in range(10):
                xn = x + step * dx
                xn[self.nb - 1:] = np.maximum(xn[self.nb - 1:], 0.3)
                cn = cost(xn)
                if np.isfinite(cn) and cn <= c * (1 + 1e-12):
                    break
                step *= 0.5
            x, c = xn, cn
            if np.max(np.abs(step * dx)) < tol:
                return Fit(x, True, it + 1)
        return Fit(x, False, max_iter)

    def huber(self, z, active, x0=None, k=1.5, iters=8) -> Fit:
        sig = self.sigma
        sig_eff = sig.copy()
        x = x0
        fit = None
        for _ in range(iters):
            fit = self.wls(z, active, sig_eff, x0=x)
            if not fit.converged:
                return fit
            x = fit.x
            r = np.abs(z - self.h(x)) / sig
            wgt = np.minimum(1.0, k / np.maximum(r, 1e-9))
            new = sig / np.sqrt(wgt)
            if np.max(np.abs(new - sig_eff) / sig_eff) < 1e-3:
                break
            sig_eff = new
        return fit

    # -- diagnostics -------------------------------------------------------------
    def analyze(self, z, x, active) -> Analysis:
        H = self.jac(x)
        w = 1.0 / self.sigma ** 2
        Ha = H[active]
        G = Ha.T @ (Ha * w[active][:, None])
        try:
            Ginv = np.linalg.inv(G)
        except np.linalg.LinAlgError:
            Ginv = np.linalg.pinv(G)
        lam = float(np.linalg.eigvalsh(G)[0])
        Q = np.einsum("ij,jk,ik->i", H, Ginv, H)
        res = z - self.h(x)
        s2 = self.sigma ** 2
        var = np.where(active, s2 - Q, s2 + Q)
        crit = active & (var < 0.02 * s2)
        rN = res / np.sqrt(np.maximum(var, 1e-12))
        rN = np.where(crit | ~self.avail, 0.0, rN)
        return Analysis(rN=rN, crit=crit, res=res, Ginv=Ginv, lam_min=lam)

    def obs_lambda(self, x, active) -> float:
        H = self.jac(x)[active]
        G = H.T @ (H / (self.sigma[active] ** 2)[:, None])
        return float(np.linalg.eigvalsh(G)[0])

    def obs_ratio(self, x, active) -> float:
        return max(0.0, self.obs_lambda(x, active)) / self.lam_ref


# =============================================================================
# 2. Scenarios (TRUTH LIVES HERE - never passed to run_pipeline)
# =============================================================================
@dataclass
class Scenario:
    x_true: np.ndarray
    z_true: np.ndarray   # noise-free measurements
    noise: np.ndarray
    load_factor: float

    @property
    def z_clean(self):
        return self.z_true + self.noise


def make_scenarios(model: GridModel, n: int, rng, load_range=(0.8, 1.2), per_load_sd=0.05):
    """n operating points: global load swing +-20%, per-load variation, AC power flow."""
    import pandapower as pp

    net = copy.deepcopy(model.net)
    p0 = net.load.p_mw.values.copy()
    q0 = net.load.q_mvar.values.copy()
    out, tries = [], 0
    while len(out) < n and tries < 6 * n + 20:
        tries += 1
        g = rng.uniform(*load_range)
        fct = g * np.clip(1 + rng.normal(0, per_load_sd, len(p0)), 0.7, 1.3)
        net.load["p_mw"] = p0 * fct
        net.load["q_mvar"] = q0 * fct
        try:
            pp.runpp(net, calculate_voltage_angles=True, init="results", numba=False)
        except Exception:
            try:
                pp.runpp(net, calculate_voltage_angles=True, init="flat", numba=False)
            except Exception:
                continue
        V = net.res_bus.vm_pu.values.copy()
        th = np.deg2rad(net.res_bus.va_degree.values)
        x = model.pack(V, th)
        out.append(Scenario(x, model.h(x), rng.normal(0.0, model.sigma), float(g)))
    return out


# =============================================================================
# 3. Attacks (evaluation / demo only)
# =============================================================================
@dataclass
class AttackInfo:
    mode: str
    bias: np.ndarray
    idx: np.ndarray
    target_bus: int | None = None
    meta: dict = field(default_factory=dict)

    @property
    def attacked_buses(self):
        return self.meta.get("attacked_buses", [])


def _stealth_bias(model, x_true, bus, dV, dth):
    """a = h(x+c) - h(x): consistent with the AC model, invisible to residual tests
    if the attacker controls every meter in the support of `a`."""
    V, th = model.unpack(x_true)
    V, th = V.copy(), th.copy()
    V[bus] += dV
    if bus != model.slack:
        th[bus] += dth
    return model.h(model.pack(V, th)) - model.h(x_true)


def _attacked_buses(model, idx):
    return sorted({int(model.loc[i]) for i in idx})


def _cfg_default():
    return PipelineConfig()


def quick_detect(model, z, cfg=None) -> bool:
    """White-box detection stage used by the adaptive attacker: would the first
    isolation round flag anything?"""
    cfg = cfg or _cfg_default()
    avail = model.avail
    plain = model.wls(z, avail)
    hub = model.huber(z, avail, x0=plain.x if plain.converged else None, k=cfg.huber_k)
    if not hub.converged:
        return True
    ana = model.analyze(z, hub.x, avail)
    s = _robust_scale(ana.rN, avail & ~ana.crit)
    zs = np.abs(ana.rN) / s
    return bool(np.any(avail & ~ana.crit & (zs >= cfg.z_iso)))


SUPPORT_EPS = 0.1  # a meter is "in the support" if the attack must move it by >= 0.1 sigma


def apply_attack(model: GridModel, sc: Scenario, mode: str, rng, cfg=None, bus=None, size=None,
                 protected=None):
    """Returns (z_attacked, AttackInfo). `bus` (0-based) and `size` for custom modes.
    `protected`: meter indices the attacker cannot touch (hardened meters)."""
    protected = np.zeros(model.m, bool) if protected is None else np.asarray(protected, bool)
    z = sc.z_clean.copy()
    avail_idx = np.where(model.avail)[0]
    sig = model.sigma
    bias = np.zeros(model.m)
    meta: dict = {}
    target = None

    if mode == "none":
        pass
    elif mode == "random_single":
        i = int(rng.choice(avail_idx))
        bias[i] = rng.choice([-1, 1]) * rng.uniform(10, 60) * sig[i]
    elif mode == "single_small":
        i = int(rng.choice(avail_idx))
        bias[i] = rng.choice([-1, 1]) * rng.uniform(3, 8) * sig[i]
    elif mode == "multi_coordinated":
        k = int(rng.integers(2, 6))
        center = int(rng.integers(model.nb))
        nbrs = set([center]) | {int(model.t[e]) for e in np.where(model.f == center)[0]} \
            | {int(model.f[e]) for e in np.where(model.t == center)[0]}
        pool = [i for i in avail_idx if int(model.loc[i]) in nbrs]
        pick = rng.choice(pool, size=min(k, len(pool)), replace=False)
        sign_by_kind = {kd: rng.choice([-1, 1]) for kd in KINDS}
        for i in pick:
            bias[int(i)] = sign_by_kind[model.kinds[i]] * rng.uniform(3, 15) * sig[i]
        target = center
    elif mode in ("stealth_partial", "adaptive", "stealth_full", "custom_stealth"):
        b = int(rng.integers(model.nb)) if bus is None else int(bus)
        target = b
        sign = rng.choice([-1, 1])
        dV0 = sign * (rng.uniform(0.03, 0.08) if size is None else abs(size))
        if size is not None and size < 0:
            dV0 = -abs(size)
        dth0 = 0.0
        a_full = _stealth_bias(model, sc.x_true, b, dV0, dth0) * model.avail
        support = np.where(np.abs(a_full) / sig > SUPPORT_EPS)[0]
        order = np.array([i for i in support[np.argsort(-np.abs(a_full[support]) / sig[support])]
                          if not protected[i]], dtype=int)
        meta.update(support_size=int(support.size), stealth_cost=int(support.size),
                    protected_in_support=int(protected[support].sum()))
        if mode in ("stealth_full", "custom_stealth"):
            bias[order] = a_full[order]          # controls everything it is allowed to touch
            meta["controlled"] = int(len(order))
        elif mode == "stealth_partial":
            ctrl = order[: min(K_MAX, order.size)]
            bias[ctrl] = a_full[ctrl]
            meta["controlled"] = int(len(ctrl))
        else:  # adaptive: white-box against the trust rule, budget K_MAX
            ctrl = order[: min(K_MAX, order.size)]
            mask = np.zeros(model.m)
            mask[ctrl] = 1.0
            lo, hi = 0.0, 1.0
            for _ in range(7):
                s = 0.5 * (lo + hi)
                a_s = _stealth_bias(model, sc.x_true, b, s * dV0, 0.0) * mask * model.avail
                if quick_detect(model, z + a_s, cfg):
                    hi = s
                else:
                    lo = s
            bias = _stealth_bias(model, sc.x_true, b, lo * dV0, 0.0) * mask * model.avail
            meta.update(controlled=int(len(ctrl)), adaptive_scale=float(lo), dV=float(lo * dV0))
    elif mode == "custom_voltage":
        b = int(rng.integers(model.nb)) if bus is None else int(bus)
        amount = float(rng.uniform(0.05, 0.15) if size is None else size)
        bias[model.iV[b]] = amount
        target = b
    else:
        raise ValueError(f"unknown attack mode {mode}")

    bias = np.where(protected, 0.0, bias)
    idx = np.where(np.abs(bias) > 0)[0]
    meta["attacked_buses"] = _attacked_buses(model, idx)
    return z + bias, AttackInfo(mode, bias, idx, target, meta)


# =============================================================================
# 4. LIVE PIPELINE (operator-visible inputs only)
# =============================================================================
@dataclass
class PipelineConfig:
    z_iso: float = 4.5         # robust z-score at which a measurement is isolated (trust 50%)
    z_full: float = 2.5        # z-score up to which trust is 100%
    z_zero: float = 6.5        # z-score at which trust is 0%
    z_reinstate: float = 2.5
    max_rounds: int = 16       # one-at-a-time removal (largest normalized residual first)
    iso_per_round: int = 1
    max_iso_frac: float = 0.25
    huber_k: float = 1.5
    use_trust: bool = True     # ablation switch
    use_obs_guard: bool = True  # ablation switch
    obs_ratio_min: float = 0.02
    tol_pu: float = 0.01       # voltage tolerance defining an unsafe recovery
    alpha: float = 0.10        # target risk level of the calibrated bound
    caution_conf: float = 0.50
    v_limits: tuple = (0.95, 1.05)


def decide_level(bound, conf, hard_fail, cfg):
    """Gate rule (shared by the live pipeline and the batch evaluation)."""
    if hard_fail:
        return "red", "VALIDATE - operator review required"
    if bound <= cfg.tol_pu:
        return "green", "PROCEED with recovered state"
    if conf >= 100.0 * cfg.caution_conf:
        return "yellow", "CAUTION - proceed only with extra checks"
    return "red", "VALIDATE - operator review required"


def _robust_scale(rN, mask):
    v = rN[mask]
    if v.size < 5:
        return 1.0
    return max(1.0, 1.4826 * float(np.median(np.abs(v - np.median(v)))))


@dataclass
class PipelineResult:
    converged: bool
    plain_V: np.ndarray
    huber_V: np.ndarray
    rec_V: np.ndarray
    rec_x: np.ndarray
    iso: np.ndarray                 # bool mask over measurements
    blocked: list
    z_score: np.ndarray
    trust: np.ndarray               # per measurement, 0-100
    bus_trust: np.ndarray           # per bus, 0-100 (min over meters located there)
    crit: np.ndarray
    features: dict
    fvec: np.ndarray
    obs_ok: bool
    conf: float | None              # calibrated P(max voltage error <= tol), 0-100
    bound_pu: float | None          # calibrated upper bound on max voltage error
    level: str                      # green / yellow / red / grey
    label: str
    reasons: list
    ms: float
    hard_fail: bool = False
    names: list = field(default_factory=list)

    @property
    def iso_idx(self):
        return np.where(self.iso)[0]

    @property
    def iso_names(self):
        return [self.names[i] for i in self.iso_idx]


def run_pipeline(model: GridModel, z: np.ndarray, cfg: PipelineConfig | None = None,
                 calibration: "GateCalibration | None" = None) -> PipelineResult:
    """LIVE PATH. Inputs: measurements, public model, offline-calibrated gate. Nothing else."""
    cfg = cfg or PipelineConfig()
    t0 = time.perf_counter()
    avail = model.avail
    m_av = int(avail.sum())

    plain = model.wls(z, avail)
    hub = model.huber(z, avail, x0=plain.x if plain.converged else None, k=cfg.huber_k)
    converged = bool(hub.converged)
    iso = np.zeros(model.m, bool)
    z_at_iso = np.zeros(model.m)
    blocked: list = []
    z_first = None
    fit = hub if converged else plain

    if cfg.use_trust and converged:
        max_iso = int(cfg.max_iso_frac * m_av)
        for _ in range(cfg.max_rounds):
            act = avail & ~iso
            ana = model.analyze(z, fit.x, act)
            s = _robust_scale(ana.rN, act & ~ana.crit)
            zs = np.abs(ana.rN) / s
            if z_first is None:
                z_first = float(zs[act & ~ana.crit].max())
            cand = np.where(act & ~ana.crit & (zs >= cfg.z_iso))[0]
            if cand.size == 0:
                break
            changed = False
            took = 0
            for i in cand[np.argsort(-zs[cand])]:
                if iso.sum() >= max_iso or took >= cfg.iso_per_round:
                    break
                trial = iso.copy()
                trial[i] = True
                if cfg.use_obs_guard and model.obs_ratio(fit.x, avail & ~trial) < cfg.obs_ratio_min:
                    if int(i) not in blocked:
                        blocked.append(int(i))
                    continue
                iso = trial
                z_at_iso[i] = zs[i]
                changed = True
                took += 1
            if not changed:
                break
            new = model.wls(z, avail & ~iso, x0=fit.x)
            if not new.converged:
                break
            fit = new
        # reinstate measurements that agree with the cleaned estimate
        if iso.any():
            act = avail & ~iso
            ana = model.analyze(z, fit.x, act)
            s = _robust_scale(ana.rN, act & ~ana.crit)
            back = [i for i in np.where(iso)[0] if abs(ana.rN[i]) / s < cfg.z_reinstate]
            if back:
                iso[back] = False
                new = model.wls(z, avail & ~iso, x0=fit.x)
                if new.converged:
                    fit = new
                else:
                    iso[back] = True

    act = avail & ~iso
    ana = model.analyze(z, fit.x, act)
    s_f = _robust_scale(ana.rN, act & ~ana.crit)
    zs_f = np.abs(ana.rN) / s_f
    z_score = np.where(iso & (z_at_iso > 0), z_at_iso, zs_f)
    trust = 100.0 * np.clip((cfg.z_zero - z_score) / (cfg.z_zero - cfg.z_full), 0.0, 1.0)
    trust = np.where(model.avail, trust, np.nan)
    bus_trust = np.array([np.nanmin(trust[(model.loc == b) & model.avail]) for b in range(model.nb)])

    r = (z - model.h(fit.x))[act] / model.sigma[act]
    df = max(int(act.sum()) - model.n_state, 1)
    J_norm = float(np.sum(r * r) / df)
    Vp, _ = model.unpack(plain.x)
    Vh, _ = model.unpack(hub.x)
    Vr, _ = model.unpack(fit.x)
    shift = float(np.max(np.abs(Vr - Vp))) if plain.converged else 1.0
    disagree = float(np.max(np.abs(Vr - Vh))) if converged else 1.0
    sdV = float(np.sqrt(np.max(np.diag(ana.Ginv)[model.stateV])))
    ratio = max(ana.lam_min, 0.0) / model.lam_ref
    obs_ok = ratio >= cfg.obs_ratio_min
    if plain.converged:
        r_raw = (z - model.h(plain.x))[avail] / model.sigma[avail]
        J_raw = float(np.sum(r_raw * r_raw) / max(m_av - model.n_state, 1))
    else:
        J_raw = 1e3
    if z_first is None:
        z_first = float(np.max(zs_f[act & ~ana.crit])) if np.any(act & ~ana.crit) else 0.0
    feats = {
        "log_J": math.log10(J_norm + 1e-3),
        "log_zmax": math.log10(float(np.max(zs_f[act & ~ana.crit])) + 0.1) if np.any(act & ~ana.crit) else 0.0,
        "frac_iso": float(iso.sum() / m_av),
        "log_shift": math.log10(shift + 1e-5),
        "log_disagree": math.log10(disagree + 1e-5),
        "log_sdV": math.log10(sdV + 1e-6),
        "n_crit": float(ana.crit.sum()),
        "obs_blocked": 1.0 if blocked else 0.0,
        "log_obs": math.log10(ratio + 1e-6),
        "log_J_raw": math.log10(J_raw + 1e-3),
        "log_zfirst": math.log10(z_first + 0.1),
    }
    fvec = np.array([feats[n] for n in FEATURE_NAMES])

    conf = bound = None
    level, label, reasons = "grey", "NOT CALIBRATED", []
    hard_fail = (not converged) or (not fit.converged) or (not obs_ok) or bool(blocked)
    if calibration is not None:
        conf = 100.0 * calibration.prob_within(fvec, cfg.tol_pu)
        bound = calibration.bound(fvec, cfg.alpha)
        level, label = decide_level(bound, conf, hard_fail, cfg)
    if not converged:
        reasons.append("State estimation did not converge: input data is inconsistent.")
    if blocked:
        reasons.append(f"{len(blocked)} suspicious measurement(s) could not be removed without losing observability.")
    if not obs_ok:
        reasons.append("Remaining measurements do not make the grid observable.")
    if iso.any():
        reasons.append("Isolated: " + ", ".join(model.names[i] for i in np.where(iso)[0][:6])
                       + (" ..." if iso.sum() > 6 else ""))
    else:
        reasons.append("No measurement isolated (all residuals within the noise level).")
    reasons.append(f"Recovery changed the voltage estimate by up to {shift:.4f} pu versus the raw estimate.")
    if bound is not None:
        reasons.append(f"Calibrated risk bound on the voltage error: {bound:.4f} pu "
                       f"(tolerance {cfg.tol_pu} pu, target risk {cfg.alpha:.0%}).")

    return PipelineResult(
        converged=converged, plain_V=Vp, huber_V=Vh, rec_V=Vr, rec_x=fit.x, iso=iso, blocked=blocked,
        z_score=z_score, trust=trust, bus_trust=bus_trust, crit=ana.crit, features=feats, fvec=fvec,
        obs_ok=bool(obs_ok), conf=conf, bound_pu=bound, level=level, label=label, reasons=reasons,
        ms=(time.perf_counter() - t0) * 1000.0, hard_fail=bool(hard_fail), names=model.names,
    )


# =============================================================================
# 5. Calibrated gate (boosted trees + split-conformal, Mondrian bins on the prediction)
# =============================================================================
class BoostedTrees:
    """Tiny gradient-boosted regression trees (numpy only, JSON-serialisable)."""

    def __init__(self, n_rounds=120, lr=0.08, depth=3, min_leaf=10, n_thr=20, subsample=0.8, seed=0):
        self.n_rounds, self.lr, self.depth = n_rounds, lr, depth
        self.min_leaf, self.n_thr, self.subsample, self.seed = min_leaf, n_thr, subsample, seed
        self.init = 0.0
        self.trees: list = []

    def _build(self, X, r, depth):
        n = len(r)
        if depth == 0 or n < 2 * self.min_leaf:
            return {"v": float(r.mean()) if n else 0.0}
        best = None
        tot, tot_n = r.sum(), n
        for j in range(X.shape[1]):
            col = X[:, j]
            thr = np.unique(np.quantile(col, np.linspace(0, 1, self.n_thr + 2)[1:-1]))
            for t in thr:
                left = col <= t
                nl = int(left.sum())
                if nl < self.min_leaf or n - nl < self.min_leaf:
                    continue
                sl = r[left].sum()
                gain = sl * sl / nl + (tot - sl) ** 2 / (tot_n - nl)
                if best is None or gain > best[0]:
                    best = (gain, j, float(t))
        if best is None:
            return {"v": float(r.mean())}
        _, j, t = best
        left = X[:, j] <= t
        return {"f": j, "t": t, "l": self._build(X[left], r[left], depth - 1),
                "r": self._build(X[~left], r[~left], depth - 1)}

    @staticmethod
    def _walk(node, x):
        while "v" not in node:
            node = node["l"] if x[node["f"]] <= node["t"] else node["r"]
        return node["v"]

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y, float)
        rng = np.random.default_rng(self.seed)
        self.init = float(y.mean())
        pred = np.full(len(y), self.init)
        self.trees = []
        for _ in range(self.n_rounds):
            idx = rng.choice(len(y), max(int(self.subsample * len(y)), 2 * self.min_leaf), replace=False)
            tree = self._build(X[idx], (y - pred)[idx], self.depth)
            self.trees.append(tree)
            pred += self.lr * np.array([self._walk(tree, x) for x in X])
        return self

    def predict_one(self, x):
        x = np.asarray(x, float)
        return self.init + self.lr * sum(self._walk(t, x) for t in self.trees)

    def to_dict(self):
        return {"init": self.init, "lr": self.lr, "trees": self.trees}

    @classmethod
    def from_dict(cls, d):
        m = cls(lr=d["lr"])
        m.init, m.trees = d["init"], d["trees"]
        return m


class GateCalibration:
    """Offline: boosted trees predict log10(max voltage error) from RUNTIME features;
    split-conformal residual quantiles (per bin of the prediction) turn the prediction
    into a bound U with P(error <= U) >= 1 - alpha under the calibration distribution.
    Online: only the feature vector is needed - no truth."""

    def __init__(self, model, edges, bin_res, tol, alpha, meta=None):
        self.model = model
        self.edges = np.asarray(edges, float)
        self.bin_res = [np.sort(np.asarray(r, float)) for r in bin_res]
        self.tol = float(tol)
        self.alpha = float(alpha)
        self.meta = meta or {}

    @classmethod
    def fit(cls, X_train, err_train, X_cal, err_cal, tol=0.01, alpha=0.10, n_bins=3, seed=0):
        X_train, X_cal = np.asarray(X_train, float), np.asarray(X_cal, float)
        y_tr = np.log10(np.maximum(np.asarray(err_train, float), 1e-6))
        y_ca = np.log10(np.maximum(np.asarray(err_cal, float), 1e-6))
        bt = BoostedTrees(seed=seed).fit(X_train, y_tr)
        yhat_cal = np.array([bt.predict_one(x) for x in X_cal])
        res = y_ca - yhat_cal
        n_bins = max(1, min(n_bins, len(res) // 80))
        edges = np.quantile(yhat_cal, np.linspace(0, 1, n_bins + 1)[1:-1]) if n_bins > 1 else np.array([])
        bidx = np.searchsorted(edges, yhat_cal)
        bin_res = [res[bidx == b] for b in range(n_bins)]
        meta = {"n_train": int(len(y_tr)), "n_cal": int(len(y_ca)), "n_bins": int(n_bins)}
        return cls(bt, edges, bin_res, tol, alpha, meta)

    def predict_log(self, fvec):
        return float(self.model.predict_one(fvec))

    def _bin(self, yhat):
        return int(np.searchsorted(self.edges, yhat))

    def prob_within(self, fvec, tol=None):
        tol = self.tol if tol is None else tol
        yhat = self.predict_log(fvec)
        R = self.bin_res[self._bin(yhat)]
        return float(np.sum(R <= math.log10(tol) - yhat) / (len(R) + 1))

    def bound(self, fvec, alpha=None):
        alpha = self.alpha if alpha is None else alpha
        yhat = self.predict_log(fvec)
        R = self.bin_res[self._bin(yhat)]
        k = math.ceil((len(R) + 1) * (1 - alpha))
        if k > len(R):
            return float("inf")
        return float(10 ** (yhat + R[k - 1]))

    def to_json(self):
        return json.dumps({
            "model": self.model.to_dict(), "edges": self.edges.tolist(),
            "bin_res": [r.tolist() for r in self.bin_res], "tol": self.tol, "alpha": self.alpha,
            "meta": self.meta, "features": list(FEATURE_NAMES),
        })

    @classmethod
    def from_json(cls, s):
        d = json.loads(s)
        if d.get("features") != list(FEATURE_NAMES):
            raise ValueError("Calibration file does not match the current feature set; re-run batch_eval.py")
        return cls(BoostedTrees.from_dict(d["model"]), d["edges"], d["bin_res"], d["tol"], d["alpha"], d.get("meta"))


# =============================================================================
# 6. Evaluation helpers (use the truth: demo / batch only)
# =============================================================================
def true_error(model, x_true, V_est) -> float:
    Vt, _ = model.unpack(x_true)
    return float(np.max(np.abs(np.asarray(V_est) - Vt)))


def limit_violations(V, limits=(0.95, 1.05)):
    V = np.asarray(V)
    return sorted(int(b) for b in np.where((V < limits[0]) | (V > limits[1]))[0])


def recovery_table(model, res: PipelineResult, z, x_true, info: AttackInfo | None, cfg=None):
    """Recovery Accuracy Evaluation (EVALUATION ONLY - never used by the live path).
    Columns: true voltage, attacked reading, naive WLS estimate, reconstructed voltage and errors."""
    cfg = cfg or PipelineConfig()
    Vt, _ = model.unpack(x_true)
    attacked = set(info.attacked_buses) if info is not None else set()
    iso_buses = {int(model.loc[i]) for i in res.iso_idx}
    rows = []
    for b in range(model.nb):
        role = "ATTACKED+ISOLATED" if (b in attacked and b in iso_buses) else \
            "ATTACKED" if b in attacked else "ISOLATED (false alarm)" if b in iso_buses else "-"
        rows.append({
            "Bus": f"B{b + 1}",
            "True voltage (pu)": round(float(Vt[b]), 4),
            "Attacked reading (pu)": round(float(z[model.iV[b]]), 4),
            "Naive estimate (pu)": round(float(res.plain_V[b]), 4),
            "Reconstructed (pu)": round(float(res.rec_V[b]), 4),
            "Reconstruction error (pu)": round(float(abs(res.rec_V[b] - Vt[b])), 4),
            "Naive error (pu)": round(float(abs(res.plain_V[b] - Vt[b])), 4),
            "Role": role,
        })
    err = np.abs(res.rec_V - Vt)
    naive = np.abs(res.plain_V - Vt)
    mx = float(err.max())
    if mx <= cfg.tol_pu:
        note = f"Recovery is close to the reference (max error {mx:.4f} pu <= {cfg.tol_pu} pu)."
    elif mx <= 3 * cfg.tol_pu:
        note = f"Recovery is acceptable but not tight (max error {mx:.4f} pu)."
    else:
        note = f"Recovery deviates strongly from the reference (max error {mx:.4f} pu): do not rely on it."
    return rows, {"max_error": mx, "mean_error": float(err.mean()), "naive_max_error": float(naive.max()),
                  "naive_mean_error": float(naive.mean()), "note": note}


def consequence_view(model, res: PipelineResult, x_true, cfg=None):
    """What the operator would conclude from voltage-limit checks on each state."""
    cfg = cfg or PipelineConfig()
    Vt, _ = model.unpack(x_true)
    t = limit_violations(Vt, cfg.v_limits)
    n = limit_violations(res.plain_V, cfg.v_limits)
    r = limit_violations(res.rec_V, cfg.v_limits)
    lab = lambda s: ", ".join(f"B{b + 1}" for b in s) or "none"
    return {
        "true": lab(t), "naive": lab(n), "recovered": lab(r),
        "naive_wrong_action": n != t, "recovered_wrong_action": r != t,
    }


def run_gridsentinel(seed=None, attack_mode="random_single", model=None, calibration=None,
                     cfg=None, bus=None, size=None):
    """Convenience API for demos and notebooks. Returns operator-view and evaluation-view keys."""
    model = model or GridModel()
    cfg = cfg or PipelineConfig()
    rng = np.random.default_rng(seed)
    sc = make_scenarios(model, 1, rng)[0]
    z, info = apply_attack(model, sc, attack_mode, rng, cfg, bus=bus, size=size)
    if calibration is None:
        calibration = get_default_calibration(model, cfg=cfg)
    res = run_pipeline(model, z, cfg, calibration)        # <- live path: z only
    rows, summary = recovery_table(model, res, z, sc.x_true, info, cfg)
    return {
        # operator view
        "level": res.level, "label": res.label, "conf": res.conf, "bound_pu": res.bound_pu,
        "isolated": res.iso_names, "success": res.converged, "residual_norm": res.features["log_J"],
        "reasons": res.reasons, "bus_trust": res.bus_trust,
        # evaluation view (demo only)
        "attack_mode": attack_mode, "attack_buses": [b + 1 for b in info.attacked_buses],
        "recovery_rows": rows, "recovery": summary, "consequence": consequence_view(model, res, sc.x_true, cfg),
        "true_error": true_error(model, sc.x_true, res.rec_V),
        "result": res, "scenario": sc, "attack": info, "z": z,
    }


# =============================================================================
# 7. Calibration builder + hardening ranking
# =============================================================================
def build_calibration(model, n_train=300, n_cal=300, seed=7, cfg=None, mix=None, progress=None):
    """Generate attacked scenarios, run the LIVE pipeline, and fit the gate offline."""
    cfg = cfg or PipelineConfig()
    mix = mix or IN_MODEL_MIX
    rng = np.random.default_rng(seed)
    modes = list(mix.keys())
    probs = np.array([mix[k] for k in modes], float)
    probs /= probs.sum()
    feats, errs = [], []
    for k, n in enumerate((n_train, n_cal)):
        F, E = [], []
        for sc in make_scenarios(model, n, rng):
            mode = str(rng.choice(modes, p=probs))
            z, _ = apply_attack(model, sc, mode, rng, cfg)
            res = run_pipeline(model, z, cfg)
            F.append(res.fvec)
            E.append(true_error(model, sc.x_true, res.rec_V))
            if progress:
                progress(len(F) + k * n_train, n_train + n_cal)
        feats.append(np.array(F))
        errs.append(np.array(E))
    return GateCalibration.fit(feats[0], errs[0], feats[1], errs[1], cfg.tol_pu, cfg.alpha)


def get_default_calibration(model, path="gate_calibration.json", cfg=None, quick_n=240, seed=7):
    import os
    if os.path.exists(path):
        try:
            return GateCalibration.from_json(open(path).read())
        except Exception:
            pass
    cal = build_calibration(model, quick_n, quick_n, seed, cfg)
    try:
        open(path, "w").write(cal.to_json())
    except OSError:
        pass
    return cal


def stealth_costs(model: GridModel, x=None):
    """For each bus: meters an attacker must control to shift that bus's voltage
    stealthily (single-bus, voltage-targeting). Lower = easier to attack."""
    x = model.x_base if x is None else x
    out = []
    for b in range(model.nb):
        a = _stealth_bias(model, x, b, 0.03, 0.0) * model.avail
        out.append(np.where(np.abs(a) / model.sigma > SUPPORT_EPS)[0])
    return out


def hardening_plan(model: GridModel):
    """Greedy hitting set: fewest meters whose protection blocks every single-bus
    voltage-targeting stealth attack (the attacker must control ALL meters in the support)."""
    sets = [set(map(int, s)) for s in stealth_costs(model)]
    remaining = list(range(len(sets)))
    chosen = []
    while remaining:
        counts = {}
        for b in remaining:
            for i in sets[b]:
                counts[i] = counts.get(i, 0) + 1
        best = max(counts, key=lambda i: (counts[i], -i))
        chosen.append((best, sorted(b for b in remaining if best in sets[b])))
        remaining = [b for b in remaining if best not in sets[b]]
    return [(int(i), model.names[i], [f"B{b + 1}" for b in covered]) for i, covered in chosen]


In [ ]:
%%writefile gridsentinel_app.py
"""GridSentinel AI - Streamlit demo.  Run:  streamlit run gridsentinel_app.py"""
import os
import numpy as np
import pandas as pd
import streamlit as st

from gridsentinel_core import (GridModel, PipelineConfig, GateCalibration, make_scenarios, apply_attack,
                               run_pipeline, recovery_table, consequence_view, hardening_plan,
                               get_default_calibration, true_error, K_MAX)

st.set_page_config(page_title="GridSentinel AI", page_icon="⚡", layout="wide")
CAL_PATH = os.environ.get("GS_CAL", "gate_calibration.json")


@st.cache_resource(show_spinner="Loading grid model and calibrated gate (first run builds it, ~1 min)...")
def load():
    model, cfg = GridModel(), PipelineConfig()
    return model, cfg, get_default_calibration(model, path=CAL_PATH, cfg=cfg)


model, cfg, cal = load()
COLORS = {"green": "#1b8a3a", "yellow": "#c98a00", "red": "#c0392b", "grey": "#666"}

st.title("⚡ GridSentinel AI")
st.caption("Self-protecting state estimation for smart grids: detect, isolate, recover - and say when NOT to trust the result. "
           "IEEE 14-bus simulation, 122 AC measurements.")

with st.sidebar:
    st.header("Operator panel")
    seed = st.number_input("Seed", min_value=0, max_value=10**9, value=42, step=1)
    fixed = st.checkbox("Fixed seed", value=True, help="Unchecked = a new random scenario on every run")
    scenario = st.selectbox("Scenario", ["none", "random single", "multi", "stealth (partial)",
                                         "adaptive attacker", "custom voltage injection", "custom stealth"])
    bus = size = None
    if scenario.startswith("custom"):
        bus = st.number_input("Target bus", 1, model.nb, 9) - 1
        size = st.number_input("Size (pu)", -0.2, 0.2, 0.08, 0.01)
    run = st.button("Run scenario", type="primary")

MODE = {"none": "none", "random single": "random_single", "multi": "multi_coordinated",
        "stealth (partial)": "stealth_partial", "adaptive attacker": "adaptive",
        "custom voltage injection": "custom_voltage", "custom stealth": "custom_stealth"}[scenario]

if "case" not in st.session_state or run:
    s = int(seed) if fixed else int(np.random.default_rng().integers(10**9))
    rng = np.random.default_rng(s)
    sc = make_scenarios(model, 1, rng)[0]
    z, info = apply_attack(model, sc, MODE, rng, cfg, bus=bus, size=size)
    res = run_pipeline(model, z, cfg, cal)          # LIVE path: z only
    st.session_state.case = dict(seed=s, mode=MODE, sc=sc, z=z, info=info, res=res)
c = st.session_state.case
res, sc, z, info = c["res"], c["sc"], c["z"], c["info"]

# --- Attack status (separate from the Trust Gate) -------------------------------------------
from scipy.stats import chi2 as _chi2
from gridsentinel_core import FEATURE_NAMES
_dof = int(model.avail.sum() - len(model.flat_start()))
_j_raw = 10 ** res.features["log_J_raw"] if "log_J_raw" in res.features else 10 ** res.fvec[FEATURE_NAMES.index("log_J_raw")]
_j_thr = _chi2.ppf(0.99, _dof) / _dof
if len(res.iso_idx) or res.blocked:
    st_txt, st_col = f"ATTACK / BAD DATA DETECTED - {len(res.iso_idx)} meter(s) isolated", "#c0392b"
elif not res.converged or _j_raw > _j_thr:
    st_txt, st_col = "SUSPICIOUS DATA - measurements disagree, no single meter to blame", "#c98a00"
else:
    st_txt, st_col = "NO ATTACK INDICATORS - all measurements are consistent", "#1b8a3a"
st.markdown(f"<div style='background:{st_col};color:white;padding:10px 18px;border-radius:10px;font-size:1.15rem;margin-bottom:8px'>"
            f"<b>Attack status:</b> {st_txt}</div>", unsafe_allow_html=True)

col = COLORS[res.level]
st.markdown(f"<div style='background:{col};color:white;padding:14px 18px;border-radius:10px;font-size:1.3rem'>"
            f"<b>{res.label}</b> &nbsp;|&nbsp; P(error ≤ {cfg.tol_pu} pu) = {res.conf:.0f}% &nbsp;|&nbsp; "
            f"calibrated error bound = {res.bound_pu:.4f} pu</div>", unsafe_allow_html=True)
if st_col == "#1b8a3a" and res.level != "green":
    st.info("Why not PROCEED? No alarm was raised, but a perfectly stealthy attack would look exactly like this. "
            "The Trust Gate only says PROCEED when it can bound the error from calibrated evidence - that is the point: "
            "it refuses to be overconfident.")
m1, m2, m3, m4 = st.columns(4)
m1.metric("Estimator converged", "yes" if res.converged else "NO")
m2.metric("Meters isolated", len(res.iso_idx))
m3.metric("Residual (chi²/dof, log10)", f"{res.features['log_J']:.2f}")
m4.metric("Runtime", f"{res.ms:.0f} ms")
st.caption(f"Scenario: {c['mode']} | seed {c['seed']}  (the operator view above uses measurements only)")

t1, t2, t3, t4 = st.tabs(["Sensor trust", "Recovery Accuracy Evaluation", "Consequence", "Hardening"])

with t1:
    st.subheader("Trust per sensor station (S01-S14)")
    tb = pd.DataFrame({"Sensor": [f"S{b + 1:02d}" for b in range(model.nb)],
                       "Bus": [f"B{b + 1}" for b in range(model.nb)],
                       "Trust %": np.round(res.bus_trust, 0)})
    tb["Status"] = np.where(tb["Trust %"] >= 75, "trusted", np.where(tb["Trust %"] >= 40, "suspect", "isolated"))
    st.dataframe(tb, hide_index=True, use_container_width=True)
    if len(res.iso_idx):
        st.write("Isolated measurements:", ", ".join(res.iso_names))
    with st.expander("Measurement-level detail"):
        d = pd.DataFrame({"Measurement": res.names, "Trust %": np.round(res.trust, 0), "z-score": np.round(res.z_score, 2),
                          "Isolated": res.iso})
        st.dataframe(d.sort_values("Trust %").head(40), hide_index=True, use_container_width=True)

with t2:
    st.subheader("Recovery Accuracy Evaluation")
    st.warning("EVALUATION ONLY. The true voltage is stored only to score the result. "
               "It is never used by the system in any live decision.")
    rows, summ = recovery_table(model, res, z, sc.x_true, info, cfg)
    df = pd.DataFrame(rows)
    st.dataframe(df, hide_index=True, use_container_width=True)
    a, b, cc = st.columns(3)
    a.metric("Max reconstruction error", f"{summ['max_error']:.4f} pu", f"naive {summ['naive_max_error']:.4f}", delta_color="off")
    b.metric("Mean reconstruction error", f"{summ['mean_error']:.4f} pu")
    cc.metric("Improvement vs naive", f"{summ['naive_max_error'] / max(summ['max_error'], 1e-9):.1f}x")
    (st.success if summ["max_error"] <= cfg.tol_pu else st.info if summ["max_error"] <= 3 * cfg.tol_pu else st.error)(summ["note"])
    st.line_chart(df.set_index("Bus")[["True voltage (pu)", "Attacked reading (pu)", "Reconstructed (pu)"]])

with t3:
    cv = consequence_view(model, res, sc.x_true, cfg)
    st.subheader("What the operator would conclude (voltage limits 0.95-1.05 pu)")
    st.table(pd.DataFrame({"State": ["True grid", "Naive estimator", "GridSentinel recovered"],
                           "Buses outside limits": [cv["true"], cv["naive"], cv["recovered"]],
                           "Wrong action?": ["-", "YES" if cv["naive_wrong_action"] else "no",
                                             "YES" if cv["recovered_wrong_action"] else "no"]}))

with t4:
    st.subheader("Hardening ranking")
    st.write(f"Attacker budget modelled: up to {K_MAX} compromised meters. Protecting these meters makes every single-bus "
             "stealth attack require control of a protected meter:")
    st.dataframe(pd.DataFrame([{"Meter to protect": n, "Covers buses": ", ".join(cv_)} for _, n, cv_ in hardening_plan(model)]),
                 hide_index=True, use_container_width=True)
    st.caption("Limits: perfect stealth beyond the budget cannot be detected by any estimator-side method.")


In [ ]:
%%writefile batch_eval.py
#!/usr/bin/env python
"""GridSentinel AI - batch evaluation (reproducible numbers for the booth / Q&A).

Usage:
    python batch_eval.py --n 300 --seed 7 --out results          # full run
    python batch_eval.py --quick --out results_quick             # smoke test (~ few minutes)
    python batch_eval.py --n 300 --no-ablation

Everything here is SIMULATION on the IEEE 14-bus system with synthetic attacks.
Truth (x_true) is used ONLY to score results - never inside run_pipeline().
Outputs (in --out): results.csv, results_summary.md, summary.json, gate_calibration.json,
                    plots: calibration.png, risk_coverage.png, error_before_after.png, ablation.png
"""
import argparse
import json
import os
import time
import warnings

import numpy as np

warnings.filterwarnings("ignore")

from gridsentinel_core import (  # noqa: E402
    IN_MODEL_MIX, FEATURE_NAMES, GateCalibration, GridModel, PipelineConfig,
    apply_attack, decide_level, hardening_plan, limit_violations, make_scenarios,
    run_pipeline, true_error,
)

TYPES = list(IN_MODEL_MIX)


# --------------------------------------------------------------------------- data generation
def gen_cases(model, cfg, n, rng, mix=None, protected=None, keep_z=False, fixed_mode=None):
    """Attack -> LIVE pipeline (z only) -> record truth-based scores separately."""
    mix = mix or IN_MODEL_MIX
    modes = list(mix)
    p = np.array([mix[k] for k in modes], float)
    p /= p.sum()
    out = []
    for sc in make_scenarios(model, n, rng):
        mode = fixed_mode or str(rng.choice(modes, p=p))
        z, info = apply_attack(model, sc, mode, rng, cfg, protected=protected)
        res = run_pipeline(model, z, cfg)
        Vt, _ = model.unpack(sc.x_true)
        att = set(int(i) for i in info.idx)
        iso = set(int(i) for i in res.iso_idx)
        out.append({
            "mode": mode, "z": z if keep_z else None, "res": res,
            "err_rec": true_error(model, sc.x_true, res.rec_V),
            "err_plain": true_error(model, sc.x_true, res.plain_V),
            "err_huber": true_error(model, sc.x_true, res.huber_V),
            "n_att": len(att), "hits": len(att & iso), "n_iso": len(iso), "false_iso": len(iso - att),
            "wrong_rec": limit_violations(res.rec_V, cfg.v_limits) != limit_violations(Vt, cfg.v_limits),
            "wrong_naive": limit_violations(res.plain_V, cfg.v_limits) != limit_violations(Vt, cfg.v_limits),
            "z": z, "Vt": Vt, "ms": res.ms,
        })
    return out


def feats(cases):
    return np.array([c["res"].fvec for c in cases])


def errs(cases):
    return np.array([c["err_rec"] for c in cases])


# --------------------------------------------------------------------------- baselines
def chi2_stats(model, z, res):
    """Plain-WLS chi-square and largest normalized residual (the textbook detectors)."""
    from scipy.stats import chi2
    fit = model.wls(z, model.avail)
    ana = model.analyze(z, fit.x, model.avail)
    r = (z - model.h(fit.x)) / model.sigma
    J = float(np.sum(r[model.avail] ** 2))
    dof = int(model.avail.sum() - len(fit.x))
    return J > chi2.ppf(0.99, dof), float(np.max(np.abs(ana.rN[~ana.crit]))) if (~ana.crit).any() else 0.0


def detection_table(model, cases, iforest):
    rows = {}
    for name in ("GridSentinel (flags/isolates)", "WLS + chi-square", "WLS + LNR (>3)", "IsolationForest (raw z)"):
        rows[name] = {"tp": 0, "fp": 0, "fn": 0, "tn": 0}
    for c in cases:
        attacked = c["mode"] != "none"
        res = c["res"]
        chi_flag, lnr = chi2_stats(model, c["z"], res)
        flags = {
            "GridSentinel (flags/isolates)": res.hard_fail or c["n_iso"] > 0,
            "WLS + chi-square": bool(chi_flag),
            "WLS + LNR (>3)": lnr > 3.0,
            "IsolationForest (raw z)": bool(iforest.predict(c["z"][None, :])[0] == -1) if iforest is not None else False,
        }
        for k, f in flags.items():
            key = ("tp" if f else "fn") if attacked else ("fp" if f else "tn")
            rows[k][key] += 1
    out = {}
    for k, d in rows.items():
        prec = d["tp"] / max(d["tp"] + d["fp"], 1)
        rec = d["tp"] / max(d["tp"] + d["fn"], 1)
        out[k] = {"precision": prec, "recall": rec, "f1": 2 * prec * rec / max(prec + rec, 1e-9),
                  "fpr": d["fp"] / max(d["fp"] + d["tn"], 1)}
    return out


# --------------------------------------------------------------------------- gate metrics
def gate_metrics(cal, cases, cfg):
    F = feats(cases)
    E = errs(cases)
    U = np.array([cal.bound(f, cfg.alpha) for f in F])
    conf = np.array([100 * cal.prob_within(f, cfg.tol_pu) for f in F])
    levels = np.array([decide_level(u, cf, c["res"].hard_fail, cfg)[0] for u, cf, c in zip(U, conf, cases)])
    proceed = levels == "green"
    unsafe = E > cfg.tol_pu
    wrong = np.array([c["wrong_rec"] for c in cases])
    cov_bound = float(np.mean(E <= U))
    return {
        "n": len(cases),
        "proceed_rate": float(proceed.mean()),
        "caution_rate": float((levels == "yellow").mean()),
        "validate_rate": float((levels == "red").mean()),
        "unsafe_given_proceed": float(unsafe[proceed].mean()) if proceed.any() else float("nan"),
        "unsafe_proceed_joint": float(np.mean(proceed & unsafe)),
        "wrong_action_given_proceed": float(wrong[proceed].mean()) if proceed.any() else float("nan"),
        "bound_coverage": cov_bound,
        "unsafe_rate_all": float(unsafe.mean()),
        "per_type": {t: {"n": int(sum(c["mode"] == t for c in cases)),
                         "proceed": float(np.mean([p for p, c in zip(proceed, cases) if c["mode"] == t] or [np.nan])),
                         "unsafe_proceed": float(np.mean([(p and u) for p, u, c in zip(proceed, unsafe, cases) if c["mode"] == t] or [np.nan]))}
                     for t in sorted(set(c["mode"] for c in cases))},
        "_U": U, "_E": E, "_proceed": proceed, "_unsafe": unsafe,
    }


def risk_coverage(score, unsafe):
    """Accept cases from lowest score (most trusted) upward. Returns coverage, selective risk, AURC."""
    order = np.argsort(score)
    u = unsafe[order].astype(float)
    k = np.arange(1, len(u) + 1)
    risk = np.cumsum(u) / k
    cov = k / len(u)
    return cov, risk, float(np.mean(risk))


# --------------------------------------------------------------------------- plots
def make_plots(out, cal_stats, rc, before_after, ablation):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    E, U = cal_stats["_E"], cal_stats["_U"]
    fin = np.isfinite(U)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.loglog(np.maximum(U[fin], 1e-5), np.maximum(E[fin], 1e-5), ".", alpha=0.4, ms=4)
    lim = [1e-5, 1]
    ax.plot(lim, lim, "k--", lw=1, label="error = bound")
    ax.axhline(0.01, color="r", lw=0.8, ls=":")
    ax.set_xlabel("Calibrated bound U (pu)")
    ax.set_ylabel("True max voltage error (pu)")
    ax.set_title(f"Calibration: {100 * cal_stats['bound_coverage']:.1f}% of cases under the bound (target >= 90%)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(out, "calibration.png"), dpi=140)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4.5))
    for name, (cov, risk, aurc) in rc.items():
        ax.plot(cov, risk, label=f"{name} (AURC {aurc:.3f})")
    ax.set_xlabel("Coverage (fraction of cases auto-accepted)")
    ax.set_ylabel("Unsafe rate among accepted")
    ax.set_title("Risk-coverage: lower is better")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "risk_coverage.png"), dpi=140)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ts = list(before_after)
    x = np.arange(len(ts))
    ax.bar(x - 0.2, [before_after[t][0] for t in ts], 0.4, label="Naive WLS")
    ax.bar(x + 0.2, [before_after[t][1] for t in ts], 0.4, label="GridSentinel recovered")
    ax.axhline(0.01, color="r", ls=":", lw=0.8)
    ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels(ts, rotation=25, ha="right", fontsize=8)
    ax.set_ylabel("Median max voltage error (pu)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(out, "error_before_after.png"), dpi=140)
    plt.close(fig)

    if ablation:
        fig, ax = plt.subplots(figsize=(7, 4))
        names = list(ablation)
        ax.bar(range(len(names)), [ablation[k]["p90"] for k in names])
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(names, rotation=25, ha="right", fontsize=8)
        ax.set_yscale("log")
        ax.set_ylabel("90th percentile max voltage error (pu)")
        ax.set_title("Ablation: what each component buys")
        fig.tight_layout()
        fig.savefig(os.path.join(out, "ablation.png"), dpi=140)
        plt.close(fig)


# --------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=300, help="scenarios per split (train / calibration / test)")
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--out", default="results")
    ap.add_argument("--quick", action="store_true", help="small run for a smoke test")
    ap.add_argument("--no-ablation", action="store_true")
    a = ap.parse_args()
    n = 80 if a.quick else a.n
    os.makedirs(a.out, exist_ok=True)
    t0 = time.time()
    cfg = PipelineConfig()
    model = GridModel()
    rng = np.random.default_rng(a.seed)
    log = lambda s: print(f"[{time.time() - t0:6.0f}s] {s}", flush=True)

    # ---- deployed grid
    log(f"generating train/cal/test ({n} each) for the deployed grid")
    tr = gen_cases(model, cfg, n, rng)
    ca = gen_cases(model, cfg, n, rng)
    te = gen_cases(model, cfg, n, rng)
    cal = GateCalibration.fit(feats(tr), errs(tr), feats(ca), errs(ca), cfg.tol_pu, cfg.alpha)
    with open(os.path.join(a.out, "gate_calibration.json"), "w") as f:
        f.write(cal.to_json())
    gm = gate_metrics(cal, te, cfg)
    log(f"deployed: proceed {gm['proceed_rate']:.2f}, unsafe|proceed {gm['unsafe_given_proceed']:.3f}, bound cov {gm['bound_coverage']:.3f}")

    # ---- hardened grid (5 protected meters)
    plan = hardening_plan(model)
    mask = np.zeros(model.m, bool)
    mask[[i for i, _, _ in plan]] = True
    log(f"hardened grid: protecting {len(plan)} meters")
    htr = gen_cases(model, cfg, n, rng, protected=mask)
    hca = gen_cases(model, cfg, n, rng, protected=mask)
    hte = gen_cases(model, cfg, n, rng, protected=mask)
    hcal = GateCalibration.fit(feats(htr), errs(htr), feats(hca), errs(hca), cfg.tol_pu, cfg.alpha)
    hm = gate_metrics(hcal, hte, cfg)
    log(f"hardened: proceed {hm['proceed_rate']:.2f}, unsafe|proceed {hm['unsafe_given_proceed']:.3f}")

    # ---- out-of-model stealth_full (limit of the method)
    sf = gen_cases(model, cfg, max(n // 3, 30), rng, fixed_mode="stealth_full")
    sfm = gate_metrics(cal, sf, cfg)
    sf_h = gen_cases(model, cfg, max(n // 3, 30), rng, fixed_mode="stealth_full", protected=mask)
    sfm_h = gate_metrics(hcal, sf_h, cfg)

    # ---- baselines: detection
    iforest = None
    try:
        from sklearn.ensemble import IsolationForest
        clean = [sc.z_clean for sc in make_scenarios(model, n, rng)]
        iforest = IsolationForest(n_estimators=200, contamination=0.01, random_state=a.seed).fit(np.array(clean))
    except Exception as e:  # sklearn missing
        log(f"IsolationForest skipped ({e})")
    det = detection_table(model, te, iforest)

    # ---- recovery error: before vs after, per attack type
    before_after, rows = {}, []
    for t in TYPES:
        cs = [c for c in te if c["mode"] == t]
        if not cs:
            continue
        ep = np.array([c["err_plain"] for c in cs])
        eh = np.array([c["err_huber"] for c in cs])
        er = np.array([c["err_rec"] for c in cs])
        before_after[t] = (float(np.median(ep)), float(np.median(er)))
        na = sum(c["n_att"] for c in cs)
        hits = sum(c["hits"] for c in cs)
        niso = sum(c["n_iso"] for c in cs)
        rows.append({
            "attack": t, "n": len(cs),
            "rmse_naive_pu": float(np.sqrt(np.mean(ep ** 2))), "rmse_huber_pu": float(np.sqrt(np.mean(eh ** 2))),
            "rmse_recovered_pu": float(np.sqrt(np.mean(er ** 2))),
            "median_naive_pu": float(np.median(ep)), "median_recovered_pu": float(np.median(er)),
            "p90_recovered_pu": float(np.quantile(er, 0.9)),
            "iso_recall": hits / na if na else float("nan"), "iso_precision": hits / niso if niso else float("nan"),
            "wrong_action_naive": float(np.mean([c["wrong_naive"] for c in cs])),
            "wrong_action_recovered": float(np.mean([c["wrong_rec"] for c in cs])),
            "proceed_rate": gm["per_type"][t]["proceed"], "unsafe_proceed": gm["per_type"][t]["unsafe_proceed"],
        })

    # ---- risk-coverage vs baselines (selector = score; lower = more trusted)
    E = gm["_E"]
    unsafe = E > cfg.tol_pu
    Fte = feats(te)
    j_raw = Fte[:, FEATURE_NAMES.index("log_J_raw")]
    rc = {
        "GridSentinel gate": risk_coverage(gm["_U"], unsafe),
        "chi-square (raw J)": risk_coverage(j_raw, unsafe),
        "always-proceed": (np.array([1.0]), np.array([unsafe.mean()]), float(unsafe.mean())),
    }

    # ---- ablations (recovery error, 90th percentile, on the same attack mix)
    ablation = {}
    if not a.no_ablation:
        log("ablations")
        variants = {
            "Full GridSentinel": (model, cfg),
            "No trust weighting": (model, PipelineConfig(use_trust=False)),
            "No observability guard": (model, PipelineConfig(use_obs_guard=False)),
            "Basic meter set (no flows)": (GridModel("basic"), cfg),
        }
        for name, (m, c) in variants.items():
            cs = gen_cases(m, c, max(n, 100), np.random.default_rng(a.seed + 100))
            e = np.array([x["err_rec"] for x in cs])
            ablation[name] = {"median": float(np.median(e)), "p90": float(np.quantile(e, 0.9)),
                              "naive_p90": float(np.quantile([x["err_plain"] for x in cs], 0.9)),
                              "iso_recall": sum(x["hits"] for x in cs) / max(sum(x["n_att"] for x in cs), 1),
                              "false_iso": sum(x["false_iso"] for x in cs) / len(cs)}
        # always-proceed ("no gate")
        ablation["No gate (always proceed)"] = {"unsafe_rate": float(unsafe.mean()), "p90": ablation["Full GridSentinel"]["p90"]}

    runtime = {"median_ms": float(np.median([c["ms"] for c in te])), "p95_ms": float(np.quantile([c["ms"] for c in te], 0.95))}

    # ---- outputs
    import csv
    with open(os.path.join(a.out, "results.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    make_plots(a.out, gm, rc, before_after, ablation)

    strip = lambda d: {k: v for k, v in d.items() if not k.startswith("_")}
    summary = {"seed": a.seed, "n_per_split": n, "deployed": strip(gm), "hardened": strip(hm),
               "stealth_full_deployed": strip(sfm), "stealth_full_hardened": strip(sfm_h),
               "detection": det, "per_attack": rows, "ablation": ablation, "runtime_ms": runtime,
               "aurc": {k: v[2] for k, v in rc.items()}, "protected_meters": [p[1] for p in plan],
               "alpha": cfg.alpha, "tol_pu": cfg.tol_pu}
    with open(os.path.join(a.out, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2, default=float)

    P = lambda x: f"{100 * x:.1f}%" if np.isfinite(x) else "n/a"
    md = [f"# GridSentinel AI - batch evaluation (seed {a.seed}, {n} scenarios per split)",
          "", "Simulation on IEEE 14-bus (122 AC measurements). Synthetic attacks, attacker budget K_MAX=8 meters. "
          "Truth is used only to score results.", "",
          "## Headline (deployed grid, in-model attacks)",
          f"- Gate target: P(max voltage error <= {cfg.tol_pu} pu) >= {100 * (1 - cfg.alpha):.0f}% per bound",
          f"- Bound coverage on held-out test: **{P(gm['bound_coverage'])}**",
          f"- PROCEED share: **{P(gm['proceed_rate'])}**; unsafe among PROCEED: **{P(gm['unsafe_given_proceed'])}**; "
          f"unsafe-PROCEED over all cases: **{P(gm['unsafe_proceed_joint'])}** (always-proceed would be {P(gm['unsafe_rate_all'])})",
          f"- Wrong voltage-limit action among PROCEED: {P(gm['wrong_action_given_proceed'])}",
          f"- Runtime per snapshot: median {runtime['median_ms']:.0f} ms, p95 {runtime['p95_ms']:.0f} ms",
          "", "## Hardened grid (5 protected meters: " + ", ".join(p[1] for p in plan) + ")",
          f"- PROCEED {P(hm['proceed_rate'])}; unsafe among PROCEED {P(hm['unsafe_given_proceed'])}; bound coverage {P(hm['bound_coverage'])}",
          "", "## Limit: out-of-model full stealth (attacker controls the whole stealth support)",
          f"- Deployed: unsafe rate {P(sfm['unsafe_rate_all'])}, PROCEED {P(sfm['proceed_rate'])}, unsafe-PROCEED {P(sfm['unsafe_proceed_joint'])}",
          f"- Hardened: unsafe rate {P(sfm_h['unsafe_rate_all'])}, PROCEED {P(sfm_h['proceed_rate'])}, unsafe-PROCEED {P(sfm_h['unsafe_proceed_joint'])}",
          "- Out-of-model: the attacker controls far more than K_MAX meters. Perfect stealth leaves every residual unchanged, so no estimator-side method can detect it; "
          "the guarantee holds only inside the stated attacker budget. Hardening did NOT fix this case in our tests - we report the number as measured.",
          "", "## Detection vs baselines (attacked vs clean, deployed test set)",
          "| Method | Precision | Recall | F1 | FPR |", "|---|---|---|---|---|"]
    md += [f"| {k} | {P(v['precision'])} | {P(v['recall'])} | {P(v['f1'])} | {P(v['fpr'])} |" for k, v in det.items()]
    md += ["", "## Recovery error per attack type (max voltage error, pu)",
           "| Attack | n | RMSE naive | RMSE Huber | RMSE recovered | p90 recovered | Iso recall | Iso precision | Wrong action naive | Wrong action recovered | PROCEED | Unsafe-PROCEED |",
           "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    md += [f"| {r['attack']} | {r['n']} | {r['rmse_naive_pu']:.4f} | {r['rmse_huber_pu']:.4f} | {r['rmse_recovered_pu']:.4f} | "
           f"{r['p90_recovered_pu']:.4f} | {P(r['iso_recall'])} | {P(r['iso_precision'])} | {P(r['wrong_action_naive'])} | "
           f"{P(r['wrong_action_recovered'])} | {P(r['proceed_rate'])} | {P(r['unsafe_proceed'])} |" for r in rows]
    md += ["", "## Risk-coverage (AURC, lower is better)"] + [f"- {k}: {v[2]:.4f}" for k, v in rc.items()]
    if ablation:
        md += ["", "## Ablations (p90 max voltage error, pu)", "| Variant | median | p90 | iso recall | false isolations / case |", "|---|---|---|---|---|"]
        for k, v in ablation.items():
            if "median" in v:
                md.append(f"| {k} | {v['median']:.4f} | {v['p90']:.4f} | {P(v['iso_recall'])} | {v['false_iso']:.2f} |")
            else:
                md.append(f"| {k} | - | - | - | unsafe rate {P(v['unsafe_rate'])} |")
    md += ["", "Files: results.csv, summary.json, gate_calibration.json, calibration.png, risk_coverage.png, "
           "error_before_after.png, ablation.png", "",
           "Note: 'wrong action' compares which buses violate [0.95, 1.05] pu; it is also non-zero on clean data because several buses sit near a limit, so read it relative to the 'none' row.", "",
           "Caveats: simulated measurements and attacks on one network; results hold for the stated attack model and load range "
           "(+-20%). Re-run with your own seed and report the numbers you measure."]
    with open(os.path.join(a.out, "results_summary.md"), "w") as f:
        f.write("\n".join(md))
    log("done -> " + a.out)
    print("\n".join(md))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile test_gridsentinel.py
"""Run: python -m pytest -q test_gridsentinel.py   (or: python test_gridsentinel.py)"""
import ast, inspect
import numpy as np
import gridsentinel_core as g

M = g.GridModel()
CFG = g.PipelineConfig()


def _sc(seed=1):
    rng = np.random.default_rng(seed)
    return rng, g.make_scenarios(M, 1, rng)[0]


def test_jacobian_matches_finite_differences():
    x = M.x_base.copy()
    J = M.jac(x)
    rng = np.random.default_rng(0)
    for j in rng.choice(len(x), 6, replace=False):
        e = np.zeros(len(x)); e[j] = 1e-6
        fd = (M.h(x + e) - M.h(x - e)) / 2e-6
        assert np.allclose(J[:, j], fd, atol=1e-4, rtol=1e-4)


def test_clean_recovery_is_accurate():
    rng, sc = _sc(2)
    res = g.run_pipeline(M, sc.z_clean, CFG)
    assert res.converged and g.true_error(M, sc.x_true, res.rec_V) < 0.01


def test_big_attack_isolated_and_recovered():
    rng, sc = _sc(3)
    z, info = g.apply_attack(M, sc, "random_single", rng, CFG)
    res = g.run_pipeline(M, z, CFG)
    assert set(info.idx) <= set(res.iso_idx)
    assert g.true_error(M, sc.x_true, res.rec_V) < 0.01   # back to the noise floor after isolation


def test_full_stealth_leaves_chi_square_unchanged():
    rng, sc = _sc(4)
    z, _ = g.apply_attack(M, sc, "stealth_full", rng, CFG, bus=8)
    a = M.wls(sc.z_clean, M.avail); b = M.wls(z, M.avail)
    ra = np.sum(((sc.z_clean - M.h(a.x)) / M.sigma)[M.avail] ** 2)
    rb = np.sum(((z - M.h(b.x)) / M.sigma)[M.avail] ** 2)
    assert abs(ra - rb) / ra < 1e-3


def test_live_path_never_touches_truth():
    src = inspect.getsource(g.run_pipeline)
    tree = ast.parse(src)
    names = {n.id for n in ast.walk(tree) if isinstance(n, ast.Name)} | {n.attr for n in ast.walk(tree) if isinstance(n, ast.Attribute)}
    assert not (names & {"x_true", "z_clean", "true_error", "Scenario", "apply_attack", "AttackInfo"})
    assert list(inspect.signature(g.run_pipeline).parameters)[:2] == ["model", "z"]


def test_seed_reproducibility():
    a = g.run_gridsentinel(seed=5, attack_mode="multi_coordinated", model=M, calibration=_cal())
    b = g.run_gridsentinel(seed=5, attack_mode="multi_coordinated", model=M, calibration=_cal())
    assert a["true_error"] == b["true_error"] and a["isolated"] == b["isolated"]


_CAL = None
def _cal():
    global _CAL
    if _CAL is None:
        _CAL = g.build_calibration(M, 120, 120, seed=3)
    return _CAL


def test_gate_bound_is_finite_and_ordered():
    r = g.run_gridsentinel(seed=6, attack_mode="none", model=M, calibration=_cal())
    assert r["bound_pu"] > 0 and 0 <= r["conf"] <= 100
    assert r["level"] in ("green", "yellow", "red")


def test_conformal_coverage_sanity():
    rng = np.random.default_rng(11)
    cases = []
    for sc in g.make_scenarios(M, 120, rng):
        z, _ = g.apply_attack(M, sc, str(rng.choice(list(g.IN_MODEL_MIX))), rng, CFG)
        res = g.run_pipeline(M, z, CFG)
        cases.append((res.fvec, g.true_error(M, sc.x_true, res.rec_V)))
    cov = np.mean([e <= _cal().bound(f) for f, e in cases])
    assert cov > 0.75   # small calibration set -> loose check; batch_eval reports the real figure


def test_calibration_json_roundtrip():
    c = _cal(); c2 = g.GateCalibration.from_json(c.to_json())
    f = np.zeros(len(g.FEATURE_NAMES))
    assert abs(c.predict_log(f) - c2.predict_log(f)) < 1e-9


if __name__ == "__main__":
    import pytest, sys
    sys.exit(pytest.main(["-q", __file__]))


In [ ]:
!python -m pytest -q test_gridsentinel.py 2>&1 | tail -3

In [ ]:
from gridsentinel_core import run_gridsentinel
r = run_gridsentinel(seed=42, attack_mode='random_single')
print(r['label'], '| conf', round(r['conf']), '% | bound', round(r['bound_pu'],4), 'pu | isolated:', r['isolated'])
print(r['recovery']['note'])

## Batch evaluation (~2 min) - these are the numbers you quote

In [ ]:
!python batch_eval.py --n 300 --seed 7 --out results | tail -60
!cp results/gate_calibration.json .

In [ ]:
from IPython.display import Image, display
for f in ['calibration','risk_coverage','error_before_after','ablation']:
    display(Image('results/'+f+'.png'))

## Launch the demo UI

In [ ]:
import subprocess, time, socket, re, os, urllib.request, stat
os.chdir("/content/gridsentinel") if os.path.exists("/content/gridsentinel") else None
subprocess.run("pkill -f streamlit; pkill -f cloudflared", shell=True)
log = open("streamlit.log", "w")
subprocess.Popen(["streamlit","run","gridsentinel_app.py","--server.port","8501","--server.headless","true",
                  "--server.enableCORS","false","--server.enableXsrfProtection","false"], stdout=log, stderr=log)
# wait until Streamlit really listens (this was the cause of "connection refused")
for _ in range(120):
    try:
        socket.create_connection(("127.0.0.1", 8501), timeout=1).close(); break
    except OSError: time.sleep(1)
else:
    print(open("streamlit.log").read()); raise SystemExit("Streamlit did not start - see log above")
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    os.chmod("cloudflared", 0o755)
tun = subprocess.Popen(["./cloudflared","tunnel","--url","http://127.0.0.1:8501","--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
t0 = time.time()
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: print("OPEN THIS LINK:", m.group(0)); break
    if time.time()-t0 > 90: print("no URL yet; check connection"); break
